# KenProbe v0 training notebook

This notebook trains **KenProbe**, a compact research model that learns search/tool-use behavior, source reading, verification, and citation-grounded answering.

Run the notebook from top to bottom. Every section below tells you whether to **run**, **skip**, or **only read** it.

**Default real-training config:** 1000 steps, 5000 HotpotQA samples, 2048 sequence length.

**Smoke-test config:** 80 steps, 500 HotpotQA samples. Use this only when checking whether the notebook works.

---

## Section 0 — GPU check

**Run this first.**

You should see a Tesla T4. If this does not show a GPU, stop and reconnect the Colab T4 runtime before continuing.

In [ ]:
!nvidia-smi

---

## Section 1 — Install dependencies

**Run this once per fresh Colab runtime.**

This may take several minutes. If NumPy gets upgraded, the next Unsloth import may ask you to restart the kernel. That is normal. Restart once, then continue from Section 2 without rerunning this install cell unless a package is missing.

In [ ]:
%%capture
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
!pip install -U datasets trl accelerate bitsandbytes sentencepiece protobuf huggingface_hub pandas

---

## Section 2 — Imports and runtime check

**Always run this after every restart.**

This cell defines core imports like `json`, `torch`, `Dataset`, `load_dataset`, and `HfApi`. If you skip it, later cells will fail with errors like `NameError: json is not defined`.

In [ ]:
import os, json, random, shutil
from datetime import datetime, timezone

import torch
from datasets import Dataset, load_dataset
from huggingface_hub import login, HfApi

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

try:
    import numpy as np
    print('NumPy:', np.__version__)
except Exception as e:
    print('NumPy check failed:', repr(e))

---

## Section 3 — Hugging Face login

**Run this before uploading adapter bundles.**

If you only want to train locally inside Colab and skip upload, you can skip this for now.

Use a Hugging Face token with write access. Local downloads later should use `hf`, not `huggingface-cli`.

In [ ]:
# Uncomment and run when you want to upload artifacts to HF.
# login()

---

## Section 4 — Training config

**Run this before model loading.**

For the real v0 run, keep:

```python
MAX_STEPS = 1000
TRAIN_HOTPOT_SAMPLES = 5000
MAX_SEQ_LENGTH = 2048
```

For a quick smoke test, change to:

```python
MAX_STEPS = 80
TRAIN_HOTPOT_SAMPLES = 500
```

In [ ]:
BASE_MODEL = 'unsloth/Qwen3.5-4B'
# Fallback only if the Qwen3.5 route fails repeatedly:
# BASE_MODEL = 'unsloth/Qwen3-4B-Instruct-2507'

PROJECT_NAME = 'kenprobe'
MODEL_SLUG = 'qwen35-4b'

HF_ADAPTER_REPO = 'amaansyed27/kenprobe-adapters'
HF_GGUF_REPO = 'amaansyed27/kenprobe-gguf'

MAX_SEQ_LENGTH = 2048
MAX_STEPS = 1000
TRAIN_HOTPOT_SAMPLES = 5000

EVAL_SIZE = 0.05
SEED = 3407

# Qwen3.5 currently works more safely with 16-bit LoRA in this notebook.
LOAD_IN_4BIT = False
LOAD_IN_16BIT = True

RUN_NAME = f'{PROJECT_NAME}-{MODEL_SLUG}-lora-step{MAX_STEPS}'
print('Run:', RUN_NAME)
print('Base model:', BASE_MODEL)
print('HotpotQA samples:', TRAIN_HOTPOT_SAMPLES)
print('Max steps:', MAX_STEPS)

---

## Section 5 — Load model and attach LoRA

**Run this once after the config cell.**

This downloads/loads Qwen and attaches trainable LoRA adapters. It can take time. If it errors because NumPy was upgraded mid-session, restart the kernel and continue from Section 2.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_16bit=LOAD_IN_16BIT,
    fast_inference=False,
    full_finetuning=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

print('Model and LoRA adapter loaded.')

---

## Section 6 — Behavior prompt and formatting helpers

**Run this before creating datasets.**

This defines KenProbe's research behavior contract and the exact training format. It also uses text-only ChatML formatting to avoid Qwen3.5 processor/image-template inference errors.

In [ ]:
import json

SYSTEM_PROMPT = '''You are KenProbe, a citation-first research chatbot.

Rules:
1. For factual, current, benchmark, product, legal, medical, financial, or research questions, request search/tool use before answering.
2. Use structured tool calls in this exact XML-like wrapper:
   <tool_call>{"name":"web_search","arguments":{"query":"..."}}</tool_call>
3. After sources are provided, answer only from the sources.
4. Cite source IDs inline as [S1], [S2].
5. If sources disagree, explain the disagreement.
6. If the evidence is insufficient, say "I do not have enough evidence" and explain what is missing.
7. Be concise, accurate, and avoid unsupported claims.
'''

text_tokenizer = getattr(tokenizer, 'tokenizer', tokenizer)

def to_chatml(messages, add_generation_prompt=False):
    parts = []
    for m in messages:
        parts.append(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>")
    if add_generation_prompt:
        parts.append('<|im_start|>assistant\n')
    return '\n'.join(parts)

def chat_text(messages):
    return to_chatml(messages, add_generation_prompt=False)

def make_tool_call(query):
    payload = {'name': 'web_search', 'arguments': {'query': query}}
    return '<tool_call>' + json.dumps(payload, ensure_ascii=False) + '</tool_call>'

def make_source_block(sources):
    blocks = []
    for s in sources:
        blocks.append(f"[{s['id']}] {s['title']}\nURL: {s.get('url', 'local-dataset')}\n{s['text']}")
    return '\n\n'.join(blocks)

print('Behavior helpers ready.')

---

## Section 7 — Seed examples

**Run this before loading HotpotQA.**

These small hand-written examples teach the exact protocol: user question → tool call → source block → cited answer. The larger dataset extends these rows.

In [ ]:
seed_examples = [
    {
        'user': 'Is a local 4B model enough to beat GPT-class models at research?',
        'query': 'small language models search tools research assistant accuracy',
        'sources': [
            {'id':'S1','title':'Research-agent design note','url':'local-seed://research-agent','text':'Small models can perform well in narrow workflows when paired with retrieval, tools, verification, and citations. They are weaker than frontier models as raw general reasoning systems.'},
            {'id':'S2','title':'RAG reliability note','url':'local-seed://rag','text':'Retrieval-augmented systems reduce hallucination risk when final answers are constrained to retrieved evidence and uncertainty is stated when evidence is insufficient.'},
        ],
        'answer': 'A 4B model is unlikely to beat frontier models as a raw general model. It can still be useful as a research system if it searches, verifies, and answers only from retrieved evidence [S1][S2].'
    },
    {
        'user': 'Should I answer current benchmark questions from memory?',
        'query': 'latest AI benchmark leaderboard current model scores',
        'sources': [{'id':'S1','title':'Benchmark freshness note','url':'local-seed://benchmarks','text':'Current model benchmark results change frequently. Answers about latest scores should be verified against current leaderboards or official model pages.'}],
        'answer': 'No. Current benchmark questions should trigger search first because leaderboard positions and model scores can change quickly [S1].'
    },
]

def seed_to_rows(examples):
    out = []
    for ex in examples:
        messages = [
            {'role':'system','content':SYSTEM_PROMPT},
            {'role':'user','content':ex['user']},
            {'role':'assistant','content':make_tool_call(ex['query'])},
            {'role':'tool','content':make_source_block(ex['sources'])},
            {'role':'assistant','content':ex['answer']},
        ]
        out.append({'text': chat_text(messages), 'source':'seed_tool_use'})
    return out

rows = seed_to_rows(seed_examples)
print('Seed rows:', len(rows))
print(rows[0]['text'][:1000])

---

## Section 8 — Build HotpotQA training rows

**Run this after seed examples.**

HotpotQA gives multi-hop questions and supporting Wikipedia contexts. This cell converts it into KenProbe's tool-use format. It may take time for 5000 samples.

In [ ]:
def build_hotpot_rows(n=500):
    hotpot = load_dataset('hotpot_qa', 'distractor', split=f'train[:{n}]')
    out = []

    for ex in hotpot:
        question = ex['question']
        answer = ex['answer']
        titles = ex['context']['title']
        sentences_list = ex['context']['sentences']

        sources = []
        for i, (title, sentences) in enumerate(zip(titles[:6], sentences_list[:6])):
            text = ' '.join(sentences[:4]).replace('\n', ' ').strip()
            if not text:
                continue
            sources.append({'id':f'S{i+1}', 'title':title, 'url':'hotpotqa://wikipedia-context', 'text':text[:1200]})

        if not sources:
            continue

        citation_ids = []
        try:
            supporting_titles = set(ex['supporting_facts']['title'])
            citation_ids = [f"[{s['id']}]" for s in sources if s['title'] in supporting_titles]
        except Exception:
            pass
        if not citation_ids:
            citation_ids = ['[S1]']

        cited_answer = f"Based on the provided sources, the answer is: {answer}. Evidence: {''.join(citation_ids[:3])}"
        messages = [
            {'role':'system','content':SYSTEM_PROMPT},
            {'role':'user','content':question},
            {'role':'assistant','content':make_tool_call(question)},
            {'role':'tool','content':make_source_block(sources)},
            {'role':'assistant','content':cited_answer},
        ]
        out.append({'text': chat_text(messages), 'source':'hotpot_qa'})

    return out

hotpot_rows = build_hotpot_rows(TRAIN_HOTPOT_SAMPLES)
rows.extend(hotpot_rows)
print('Loaded HotpotQA rows:', len(hotpot_rows))
print('Total rows:', len(rows))

---

## Section 9 — Create train/eval split

**Run this after dataset rows are created.**

This builds the Hugging Face `Dataset` objects used by TRL's SFT trainer.

In [ ]:
random.seed(SEED)
random.shuffle(rows)

dataset = Dataset.from_list(rows).shuffle(seed=SEED)
split = dataset.train_test_split(test_size=EVAL_SIZE, seed=SEED)
train_dataset = split['train']
eval_dataset = split['test']

print(train_dataset)
print('Eval:', eval_dataset)
print('Preview:')
print(train_dataset[0]['text'][:2000])

---

## Section 10 — Train LoRA adapter

**This is the main training cell.**

For the default real run this trains for 1000 steps. Do not interrupt unless it errors. After it finishes, continue to Section 11 to save and bundle the adapter.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=text_tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir=f'outputs_{RUN_NAME}',
        optim='adamw_8bit',
        seed=SEED,
        dataset_num_proc=1,
        fp16=True,
        bf16=False,
        report_to='none',
    ),
)

train_result = trainer.train()
print(train_result)

---

## Section 11 — Save adapter, report, and zip bundle

**Run this immediately after training finishes.**

This saves the LoRA adapter and tokenizer files into `adapters/`, writes a JSON report into `reports/`, then creates one zip bundle in `runs/`. These folders exist on the Colab VM first. The next section uploads the zip to HF so you can download it locally with `hf`.

In [ ]:
RUN_NAME = f'{PROJECT_NAME}-{MODEL_SLUG}-lora-step{MAX_STEPS}'
ADAPTER_DIR = f'adapters/{RUN_NAME}'
REPORT_DIR = 'reports'
BUNDLE_DIR = f'runs/{RUN_NAME}'

os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(BUNDLE_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
text_tokenizer.save_pretrained(ADAPTER_DIR)

report = {
    'run_name': RUN_NAME,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'base_model': BASE_MODEL,
    'max_steps': MAX_STEPS,
    'max_seq_length': MAX_SEQ_LENGTH,
    'train_hotpot_samples': TRAIN_HOTPOT_SAMPLES,
    'eval_size': EVAL_SIZE,
    'load_in_4bit': LOAD_IN_4BIT,
    'load_in_16bit': LOAD_IN_16BIT,
    'adapter_dir': ADAPTER_DIR,
    'hf_adapter_repo': HF_ADAPTER_REPO,
}

REPORT_PATH = f'{REPORT_DIR}/{RUN_NAME}.json'
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

shutil.copytree(ADAPTER_DIR, f'{BUNDLE_DIR}/adapter', dirs_exist_ok=True)
shutil.copy(REPORT_PATH, f'{BUNDLE_DIR}/report.json')
BUNDLE_ZIP = shutil.make_archive(BUNDLE_DIR, 'zip', BUNDLE_DIR)

print('Saved adapter:', ADAPTER_DIR)
print('Saved report:', REPORT_PATH)
print('Created bundle:', BUNDLE_ZIP)
!ls -lh {BUNDLE_ZIP}

---

## Section 12 — Upload adapter bundle to HF

**Run this after Section 11 if you ran `login()` in Section 3.**

This uploads the zip bundle to `amaansyed27/kenprobe-adapters`. Later, on Windows, download it with:

```powershell
hf auth login
hf download amaansyed27/kenprobe-adapters kenprobe-qwen35-4b-lora-step1000.zip --local-dir runs
```

In [ ]:
api = HfApi()
api.create_repo(repo_id=HF_ADAPTER_REPO, repo_type='model', private=True, exist_ok=True)
api.upload_file(
    path_or_fileobj=BUNDLE_ZIP,
    path_in_repo=os.path.basename(BUNDLE_ZIP),
    repo_id=HF_ADAPTER_REPO,
    repo_type='model',
)
print('Uploaded adapter bundle to HF:')
print(f'{HF_ADAPTER_REPO}/{os.path.basename(BUNDLE_ZIP)}')

---

## Section 13 — Inference helper

**Run this before test prompts.**

This defines a safe text-only generation function. It avoids the earlier `Incorrect padding` processor/image error.

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

def generate_text(messages, max_new_tokens=512, temperature=0.2):
    prompt = to_chatml(messages, add_generation_prompt=True)
    inputs = text_tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            pad_token_id=text_tokenizer.eos_token_id,
            eos_token_id=text_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[-1]:]
    return text_tokenizer.decode(new_tokens, skip_special_tokens=True)

print('Inference helper ready.')

---

## Section 14 — Test 1: search/tool-call behavior

**Run this after training and saving.**

Expected output should contain one or more `<tool_call>{...}</tool_call>` blocks. If it gives a normal answer instead, the adapter is weak and needs more training data or steps.

In [ ]:
test_messages = [
    {'role':'system','content':SYSTEM_PROMPT},
    {'role':'user','content':'Compare Qwythos 9B with Gemini Flash for research tasks. Be accurate.'},
]
print(generate_text(test_messages, max_new_tokens=350, temperature=0.2))

---

## Section 15 — Test 2: source-grounded answer behavior

**Run this after Section 14.**

Expected behavior: the model should answer from the provided `[S1]` and `[S2]` sources and should not claim Qwythos beats Gemini Flash.

In [ ]:
source_block = '''[S1] Qwythos model card
URL: https://huggingface.co/example/qwythos
Qwythos is a 9B model based on Qwen and advertises long-context capability. Its benchmark claims are self-reported on the model card.

[S2] Public benchmark leaderboard
URL: https://example.com/leaderboard
The public leaderboard does not list Qwythos. Frontier cloud models have independent agent benchmark scores that are much higher than small local models.
'''

test_messages = [
    {'role':'system','content':SYSTEM_PROMPT},
    {'role':'user','content':'Is Qwythos proven to beat Gemini Flash?'},
    {'role':'tool','content':source_block},
]
print(generate_text(test_messages, max_new_tokens=400, temperature=0.2))

---

## Section 16 — Optional GGUF export

**Skip this for now.**

Only run after the LoRA adapter output looks useful. GGUF export can take time and may fail on Colab. When ready, set `DO_GGUF_EXPORT = True`.

In [ ]:
DO_GGUF_EXPORT = False

if DO_GGUF_EXPORT:
    GGUF_DIR = f'models/{RUN_NAME}-Q4_K_M'
    model.save_pretrained_gguf(GGUF_DIR, text_tokenizer, quantization_method='q4_k_m')
    api = HfApi()
    api.create_repo(repo_id=HF_GGUF_REPO, repo_type='model', private=True, exist_ok=True)
    for root, _, files in os.walk(GGUF_DIR):
        for name in files:
            if name.endswith('.gguf'):
                local_path = os.path.join(root, name)
                api.upload_file(path_or_fileobj=local_path, path_in_repo=name, repo_id=HF_GGUF_REPO, repo_type='model')
                print('Uploaded:', name)
else:
    print('GGUF export skipped.')

---

## Section 17 — Local download commands

**Only read this. Run these in local PowerShell after HF upload finishes.**

```powershell
cd C:\Users\Amaan\Downloads\ken-research
hf auth login
hf download amaansyed27/kenprobe-adapters kenprobe-qwen35-4b-lora-step1000.zip --local-dir runs
mkdir adapters,reports,models,runs -Force
Expand-Archive .\runs\kenprobe-qwen35-4b-lora-step1000.zip -DestinationPath .\runs\step1000 -Force
Copy-Item .\runs\step1000\adapter .\adapters\kenprobe-qwen35-4b-lora-step1000 -Recurse -Force
Copy-Item .\runs\step1000\report.json .\reports\kenprobe-qwen35-4b-lora-step1000.json -Force
```